# 03 — Controls: Permutation and Pre-Stimulus

This notebook runs two anti-overfitting controls for the selected OpenNeuro ERP/oddball within-subject stimulus-classification benchmark.

## Goals

We evaluate whether the benchmarked models are learning meaningful post-stimulus signal rather than exploiting leakage or unrelated structure.

## Controls

### 1. Permuted-label control
The EEG data remains unchanged, but labels are shuffled independently within each subject.

Expected outcome:
- performance should collapse toward chance

### 2. Pre-stimulus-only control
The benchmark is rerun using only the pre-stimulus window.

Expected outcome:
- performance should be clearly weaker than the main benchmark

## Models compared

- **LDA**
- **Logistic Regression**
- **EEGNet**

## Outputs

This notebook will produce:

- fold-level control results
- subject-level control summaries
- model-level control summaries
- direct comparison tables:
  - main vs permuted labels
  - main vs pre-stimulus-only
- comparison plots

This notebook provides the main anti-overfitting evidence for the thesis.

All outputs are saved under:

`results/<dataset>/<task>/<signal>/<eval_mode>/<control_mode>/`

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

import pandas as pd

from oddball_eeg.bids import make_epoching_config_from_experiment
from oddball_eeg.config import get_experiment_family
from oddball_eeg.io import save_dataframe, save_experiment_metadata, save_history_dict
from oddball_eeg.loader import load_subjects_grouped
from oddball_eeg.preprocessing import (
    apply_crop_to_loaded_dataset,
    apply_permuted_labels_to_loaded_dataset,
)
from oddball_eeg.reproducibility import get_default_device
from oddball_eeg.splits import build_folds_by_subject
from oddball_eeg.summaries import (
    summarize_fold_results,
    summarize_model_results_across_subjects,
    summarize_subject_results_across_seeds,
)
from oddball_eeg.training import evaluate_classical_benchmark, evaluate_eegnet_benchmark
from oddball_eeg.visualization import (
    make_main_vs_control_table,
    make_model_ranking_table,
    plot_main_vs_control,
    plot_model_ranking_bars,
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
)

In [ ]:
set_pandas_display_options()

## 1. Load configurations

We will use:

- the main benchmark configuration as the reference
- the permuted-label control configuration
- the pre-stimulus-only control configuration

In [ ]:
# Change only this line when switching datasets.
# EXPERIMENT_NAME = "ds003061_within_subject_max15"
# EXPERIMENT_NAME = "ds005863_within_subject_max15"
# EXPERIMENT_NAME = "ds006466_within_subject_max15"
EXPERIMENT_NAME = "ds006480_within_subject_max15"


experiment_family = get_experiment_family(EXPERIMENT_NAME)

cfg_main = experiment_family["main"]
cfg_perm = experiment_family["permuted"]
cfg_prestim = experiment_family["prestim"]
cfg_loso = experiment_family["loso"]

device = get_default_device()

save_experiment_metadata(
    cfg_perm,
    extra={
        "notebook": "03_controls_permutation_and_prestim.ipynb",
        "purpose": "Permuted-label anti-overfitting control.",
        "main_experiment_key": EXPERIMENT_NAME,
        "permutation_seed": 12345,
    },
)

save_experiment_metadata(
    cfg_prestim,
    extra={
        "notebook": "03_controls_permutation_and_prestim.ipynb",
        "purpose": "Pre-stimulus-only anti-overfitting control.",
        "main_experiment_key": EXPERIMENT_NAME,
    },
)

print("Experiment key        :", EXPERIMENT_NAME)
print("Dataset               :", cfg_main.dataset.dataset)
print("Task                  :", cfg_main.dataset.task)
print("Subjects              :", cfg_main.subjects.subjects)
print("Seeds                 :", cfg_main.seeds.seeds)
print("Signal config         :", cfg_main.signal)
print("Main crop             :", cfg_main.effective_crop_window())
print("Permuted control mode :", cfg_perm.control_mode)
print("Pre-stim crop         :", cfg_prestim.effective_crop_window())
print("Chunk size            :", cfg_main.splits.group_chunk_size)
print("Purge neighbors       :", cfg_main.splits.purge_neighbor_groups)
print("Device                :", device)

print("\nMain result directory:")
print(cfg_main.result_dir())

print("\nPermuted-label result directory:")
print(cfg_perm.result_dir())

print("\nPre-stimulus result directory:")
print(cfg_prestim.result_dir())

## 2. Load the shared full dataset

We load the data only once from the full time range and then derive the control datasets from it.

In [ ]:
epoching_cfg = make_epoching_config_from_experiment(cfg_main)

print(epoching_cfg)

data_full = load_subjects_grouped(
    dataset=cfg_main.dataset.dataset,
    task_name=cfg_main.dataset.task,
    subjects=cfg_main.subjects.subjects,
    epoching_cfg=epoching_cfg,
    chunk_size=cfg_main.splits.group_chunk_size,
    apply_baseline=True,
    verbose=True,
)

## 3. Build the three dataset variants

We prepare:

- `data_main`: main benchmark crop
- `data_perm`: main crop with permuted labels
- `data_prestim`: pre-stimulus-only crop

In [ ]:
main_window = cfg_main.effective_crop_window()
prestim_window = cfg_prestim.effective_crop_window()

assert main_window is not None
assert prestim_window is not None

data_main = apply_crop_to_loaded_dataset(
    data_full,
    tmin=main_window[0],
    tmax=main_window[1],
)

data_perm = apply_permuted_labels_to_loaded_dataset(
    data_main,
    seed=12345,
)

data_prestim = apply_crop_to_loaded_dataset(
    data_full,
    tmin=prestim_window[0],
    tmax=prestim_window[1],
)

## 4. Build grouped folds for each dataset

The grouped fold geometry should be built within each dataset variant using the same grouped design.

In [ ]:
folds_perm = build_folds_by_subject(
    data_perm,
    n_splits=cfg_perm.splits.n_splits,
    purge_neighbors=cfg_perm.splits.purge_neighbor_groups,
)

folds_prestim = build_folds_by_subject(
    data_prestim,
    n_splits=cfg_prestim.splits.n_splits,
    purge_neighbors=cfg_prestim.splits.purge_neighbor_groups,
)

for label, data_obj, folds_obj, cfg_obj in [
    ("permuted", data_perm, folds_perm, cfg_perm),
    ("prestim", data_prestim, folds_prestim, cfg_prestim),
]:
    rows = []

    for sub in data_obj.subjects:
        y = data_obj.y_by_sub[sub]

        for fold_id, (tr_idx, te_idx) in enumerate(folds_obj[sub], start=1):
            rows.append({
                "control": label,
                "subject": sub,
                "fold": int(fold_id),
                "n_train": int(len(tr_idx)),
                "n_test": int(len(te_idx)),
                "train_target_rate": float(y[tr_idx].mean()),
                "test_target_rate": float(y[te_idx].mean()),
            })

    df_fold_design = pd.DataFrame(rows)
    df_fold_design = round_numeric_columns(df_fold_design, decimals=4)

    display(df_fold_design)

    save_dataframe(
        df_fold_design,
        cfg_obj.table_dir("fold_design") / "outer_fold_design.csv",
    )

## 5. Load the main benchmark summaries

This notebook assumes the main benchmark notebook has already been run and saved its outputs.

We reload them here so we can compare controls directly against the main benchmark.

In [ ]:
MAIN_SUMMARY_DIR = cfg_main.table_dir("summaries")

df_main_subject_seed = pd.read_csv(MAIN_SUMMARY_DIR / "subject_seed_summary.csv")
df_main_subject_across_seeds = pd.read_csv(MAIN_SUMMARY_DIR / "subject_summary_across_seeds.csv")
df_main_model_summary = pd.read_csv(MAIN_SUMMARY_DIR / "model_summary_across_subjects.csv")

display(round_numeric_columns(df_main_model_summary, decimals=4))

## 6. Run the permuted-label control across seeds

In [ ]:
all_perm_folds = []
all_perm_histories_by_seed = {}

for seed in cfg_perm.seeds.seeds:
    print("#" * 90)
    print(f"PERMUTED LABEL CONTROL — SEED {seed}")
    print("#" * 90)

    seed_fold_dir = cfg_perm.table_dir("folds", seed=seed)
    seed_history_dir = cfg_perm.history_dir("eegnet", seed=seed)

    df_classical_perm = evaluate_classical_benchmark(
        data=data_perm,
        folds_by_sub=folds_perm,
        model_names=("lda", "logreg"),
        seed=seed,
        verbose=cfg_perm.verbose,
        feature_mode=cfg_perm.feature.mode,
        bin_ms=cfg_perm.feature.bin_ms,
    )

    save_dataframe(
        df_classical_perm,
        seed_fold_dir / "classical_fold_results.csv",
    )

    df_eegnet_perm, histories_perm = evaluate_eegnet_benchmark(
        data=data_perm,
        folds_by_sub=folds_perm,
        device=device,
        normalize=cfg_perm.eegnet.normalize,
        batch_size=cfg_perm.eegnet.batch_size,
        lr=cfg_perm.eegnet.lr,
        weight_decay=cfg_perm.eegnet.weight_decay,
        epochs=cfg_perm.eegnet.epochs,
        patience=cfg_perm.eegnet.early_stopping_patience,
        val_size=cfg_perm.splits.val_size,
        val_n_tries=cfg_perm.splits.val_n_tries,
        use_class_weight=cfg_perm.eegnet.use_class_weight,
        grad_clip_norm=cfg_perm.eegnet.grad_clip_norm,
        eegnet_kwargs={
            "n_classes": cfg_perm.eegnet.n_classes,
            "f1": cfg_perm.eegnet.f1,
            "d": cfg_perm.eegnet.d,
            "f2": cfg_perm.eegnet.f2,
            "k1": cfg_perm.eegnet.k1,
            "k2": cfg_perm.eegnet.k2,
            "dropout": cfg_perm.eegnet.dropout,
        },
        seed=seed,
        verbose=cfg_perm.verbose,
    )

    save_dataframe(
        df_eegnet_perm,
        seed_fold_dir / "eegnet_fold_results.csv",
    )

    save_history_dict(
        histories_perm,
        seed_history_dir,
        filename_prefix="eegnet_history",
    )

    df_seed_perm = pd.concat(
        [df_classical_perm, df_eegnet_perm],
        ignore_index=True,
    )

    save_dataframe(
        df_seed_perm,
        seed_fold_dir / "all_fold_results.csv",
    )

    all_perm_folds.append(df_seed_perm)
    all_perm_histories_by_seed[seed] = histories_perm

In [ ]:
PERM_SUMMARY_DIR = cfg_perm.table_dir("summaries")

df_perm_all_folds = pd.concat(all_perm_folds, ignore_index=True)

save_dataframe(
    df_perm_all_folds,
    PERM_SUMMARY_DIR / "all_fold_results_across_seeds.csv",
)

df_perm_subject_seed = summarize_fold_results(
    df_perm_all_folds,
    group_cols=("subject", "model", "seed"),
)

save_dataframe(
    df_perm_subject_seed,
    PERM_SUMMARY_DIR / "subject_seed_summary.csv",
)

df_perm_subject_across_seeds = summarize_subject_results_across_seeds(
    df_perm_subject_seed
)

save_dataframe(
    df_perm_subject_across_seeds,
    PERM_SUMMARY_DIR / "subject_summary_across_seeds.csv",
)

df_perm_model_summary = summarize_model_results_across_subjects(
    df_perm_subject_across_seeds
)

save_dataframe(
    df_perm_model_summary,
    PERM_SUMMARY_DIR / "model_summary_across_subjects.csv",
)

display(round_numeric_columns(df_perm_model_summary, decimals=4))

## 7. Run the pre-stimulus-only control across seeds

In [ ]:
all_prestim_folds = []
all_prestim_histories_by_seed = {}

for seed in cfg_prestim.seeds.seeds:
    print("#" * 90)
    print(f"PRE-STIMULUS CONTROL — SEED {seed}")
    print("#" * 90)

    seed_fold_dir = cfg_prestim.table_dir("folds", seed=seed)
    seed_history_dir = cfg_prestim.history_dir("eegnet", seed=seed)

    df_classical_prestim = evaluate_classical_benchmark(
        data=data_prestim,
        folds_by_sub=folds_prestim,
        model_names=("lda", "logreg"),
        seed=seed,
        verbose=cfg_prestim.verbose,
        feature_mode=cfg_prestim.feature.mode,
        bin_ms=cfg_prestim.feature.bin_ms,
    )

    save_dataframe(
        df_classical_prestim,
        seed_fold_dir / "classical_fold_results.csv",
    )

    df_eegnet_prestim, histories_prestim = evaluate_eegnet_benchmark(
        data=data_prestim,
        folds_by_sub=folds_prestim,
        device=device,
        normalize=cfg_prestim.eegnet.normalize,
        batch_size=cfg_prestim.eegnet.batch_size,
        lr=cfg_prestim.eegnet.lr,
        weight_decay=cfg_prestim.eegnet.weight_decay,
        epochs=cfg_prestim.eegnet.epochs,
        patience=cfg_prestim.eegnet.early_stopping_patience,
        val_size=cfg_prestim.splits.val_size,
        val_n_tries=cfg_prestim.splits.val_n_tries,
        use_class_weight=cfg_prestim.eegnet.use_class_weight,
        grad_clip_norm=cfg_prestim.eegnet.grad_clip_norm,
        eegnet_kwargs={
            "n_classes": cfg_prestim.eegnet.n_classes,
            "f1": cfg_prestim.eegnet.f1,
            "d": cfg_prestim.eegnet.d,
            "f2": cfg_prestim.eegnet.f2,
            "k1": cfg_prestim.eegnet.k1,
            "k2": cfg_prestim.eegnet.k2,
            "dropout": cfg_prestim.eegnet.dropout,
        },
        seed=seed,
        verbose=cfg_prestim.verbose,
    )

    save_dataframe(
        df_eegnet_prestim,
        seed_fold_dir / "eegnet_fold_results.csv",
    )

    save_history_dict(
        histories_prestim,
        seed_history_dir,
        filename_prefix="eegnet_history",
    )

    df_seed_prestim = pd.concat(
        [df_classical_prestim, df_eegnet_prestim],
        ignore_index=True,
    )

    save_dataframe(
        df_seed_prestim,
        seed_fold_dir / "all_fold_results.csv",
    )

    all_prestim_folds.append(df_seed_prestim)
    all_prestim_histories_by_seed[seed] = histories_prestim

In [ ]:
PRESTIM_SUMMARY_DIR = cfg_prestim.table_dir("summaries")

df_prestim_all_folds = pd.concat(all_prestim_folds, ignore_index=True)

save_dataframe(
    df_prestim_all_folds,
    PRESTIM_SUMMARY_DIR / "all_fold_results_across_seeds.csv",
)

df_prestim_subject_seed = summarize_fold_results(
    df_prestim_all_folds,
    group_cols=("subject", "model", "seed"),
)

save_dataframe(
    df_prestim_subject_seed,
    PRESTIM_SUMMARY_DIR / "subject_seed_summary.csv",
)

df_prestim_subject_across_seeds = summarize_subject_results_across_seeds(
    df_prestim_subject_seed
)

save_dataframe(
    df_prestim_subject_across_seeds,
    PRESTIM_SUMMARY_DIR / "subject_summary_across_seeds.csv",
)

df_prestim_model_summary = summarize_model_results_across_subjects(
    df_prestim_subject_across_seeds
)

save_dataframe(
    df_prestim_model_summary,
    PRESTIM_SUMMARY_DIR / "model_summary_across_subjects.csv",
)

display(round_numeric_columns(df_prestim_model_summary, decimals=4))

## 8. Compare main benchmark vs permuted-label control

In [ ]:
df_main_vs_perm = make_main_vs_control_table(
    df_main_subject_across_seeds,
    df_perm_subject_across_seeds,
    value_col_main="auc_seed_mean",
    value_col_control="auc_seed_mean",
    main_label="main",
    control_label="permuted",
)

PERM_COMPARISON_DIR = cfg_perm.table_dir("comparisons")

df_main_vs_perm = round_numeric_columns(df_main_vs_perm, decimals=4)

display(df_main_vs_perm)

save_dataframe(
    df_main_vs_perm,
    PERM_COMPARISON_DIR / "main_vs_permuted_subject_table.csv",
)

## 9. Compare main benchmark vs pre-stimulus-only control

In [ ]:
df_main_vs_prestim = make_main_vs_control_table(
    df_main_subject_across_seeds,
    df_prestim_subject_across_seeds,
    value_col_main="auc_seed_mean",
    value_col_control="auc_seed_mean",
    main_label="main",
    control_label="prestim",
)

PRESTIM_COMPARISON_DIR = cfg_prestim.table_dir("comparisons")

df_main_vs_prestim = round_numeric_columns(df_main_vs_prestim, decimals=4)

display(df_main_vs_prestim)

save_dataframe(
    df_main_vs_prestim,
    PRESTIM_COMPARISON_DIR / "main_vs_prestim_subject_table.csv",
)

## 10. Model ranking table for the permuted-label control

In [ ]:
table_perm_ranking = make_model_ranking_table(
    df_perm_model_summary,
    metric_col="mean_subject_auc",
)

display(table_perm_ranking)

save_dataframe(
    table_perm_ranking,
    cfg_perm.table_dir("reporting") / "permuted_model_ranking_table.csv",
)

## 11. Model ranking table for the pre-stimulus-only control

In [ ]:
table_prestim_ranking = make_model_ranking_table(
    df_prestim_model_summary,
    metric_col="mean_subject_auc",
)

display(table_prestim_ranking)

save_dataframe(
    table_prestim_ranking,
    cfg_prestim.table_dir("reporting") / "prestim_model_ranking_table.csv",
)

## 12. Plot main benchmark vs permuted-label control

In [ ]:
fig_main_vs_perm = plot_main_vs_control(
    df_main_vs_perm,
    main_col="main_value",
    control_col="permuted_value",
    title=f"Main benchmark vs permuted-label control — {cfg_main.dataset.dataset}, {cfg_main.dataset.task}",
)
fig_main_vs_perm

## 13. Plot main benchmark vs pre-stimulus-only control

In [ ]:
fig_main_vs_prestim = plot_main_vs_control(
    df_main_vs_prestim,
    main_col="main_value",
    control_col="prestim_value",
    title=f"Main benchmark vs pre-stimulus-only control — {cfg_main.dataset.dataset}, {cfg_main.dataset.task}",
)
fig_main_vs_prestim

## 14. Plot model ranking for the permuted-label control

In [ ]:
fig_perm_rank = plot_model_ranking_bars(
    df_perm_model_summary,
    metric_col="mean_subject_auc",
    title=f"Permuted-label control: model ranking — {cfg_main.dataset.dataset}, {cfg_main.dataset.task}",
    ylabel="Mean subject ROC-AUC",
)
fig_perm_rank

## 15. Plot model ranking for the pre-stimulus-only control

In [ ]:
fig_prestim_rank = plot_model_ranking_bars(
    df_prestim_model_summary,
    metric_col="mean_subject_auc",
    title=f"Pre-stimulus-only control: model ranking — {cfg_main.dataset.dataset}, {cfg_main.dataset.task}",
    ylabel="Mean subject ROC-AUC",
)
fig_prestim_rank

## 16. Save key control figures

In [ ]:
PERM_CONTROL_FIG_DIR = cfg_perm.figure_dir("controls")
PRESTIM_CONTROL_FIG_DIR = cfg_prestim.figure_dir("controls")

save_figure(
    fig_main_vs_perm,
    PERM_CONTROL_FIG_DIR / "main_vs_permuted.png",
)

save_figure(
    fig_perm_rank,
    PERM_CONTROL_FIG_DIR / "permuted_model_ranking.png",
)

save_figure(
    fig_main_vs_prestim,
    PRESTIM_CONTROL_FIG_DIR / "main_vs_prestim.png",
)

save_figure(
    fig_prestim_rank,
    PRESTIM_CONTROL_FIG_DIR / "prestim_model_ranking.png",
)

save_experiment_metadata(
    cfg_perm,
    extra={
        "notebook": "03_controls_permutation_and_prestim.ipynb",
        "status": "completed",
        "main_experiment_key": EXPERIMENT_NAME,
        "control": "permuted_labels",
        "n_subjects": int(len(data_perm.subjects)),
        "n_seeds": int(len(cfg_perm.seeds.seeds)),
        "models": sorted(df_perm_all_folds["model"].unique().tolist()),
        "n_fold_rows": int(len(df_perm_all_folds)),
        "summary_dir": str(PERM_SUMMARY_DIR),
        "comparison_dir": str(PERM_COMPARISON_DIR),
        "figure_dir": str(PERM_CONTROL_FIG_DIR),
    },
)

save_experiment_metadata(
    cfg_prestim,
    extra={
        "notebook": "03_controls_permutation_and_prestim.ipynb",
        "status": "completed",
        "main_experiment_key": EXPERIMENT_NAME,
        "control": "prestim_only",
        "n_subjects": int(len(data_prestim.subjects)),
        "n_seeds": int(len(cfg_prestim.seeds.seeds)),
        "models": sorted(df_prestim_all_folds["model"].unique().tolist()),
        "n_fold_rows": int(len(df_prestim_all_folds)),
        "summary_dir": str(PRESTIM_SUMMARY_DIR),
        "comparison_dir": str(PRESTIM_COMPARISON_DIR),
        "figure_dir": str(PRESTIM_CONTROL_FIG_DIR),
    },
)

print("Saved permuted-label figures to:", PERM_CONTROL_FIG_DIR)
print("Saved pre-stimulus figures to:", PRESTIM_CONTROL_FIG_DIR)

## 17. Control interpretation checklist

When reviewing the results, the expected pattern is:

### Permuted labels
- performance should move toward chance
- no model should retain strong discriminative power

### Pre-stimulus-only
- performance should be clearly weaker than the main benchmark
- models may retain some limited structure, but not competitive post-stimulus performance

If these patterns hold, they strengthen the claim that the main benchmark is learning meaningful post-stimulus information.

In [ ]:
print("=" * 80)
print("Control summary")
print("=" * 80)

print("\nMain benchmark model summary")
display(round_numeric_columns(df_main_model_summary, decimals=4))

print("\nPermuted-label control model summary")
display(round_numeric_columns(df_perm_model_summary, decimals=4))

print("\nPre-stimulus-only control model summary")
display(round_numeric_columns(df_prestim_model_summary, decimals=4))

## 18. Control conclusions

This notebook provides the core anti-overfitting evidence for the thesis.

The intended interpretation is:

- the main benchmark should outperform both controls
- permuted labels should collapse toward chance
- pre-stimulus-only performance should be materially lower than the main benchmark

If that pattern is observed, the benchmark is much more trustworthy.

The next notebook should evaluate **cross-subject generalization** using **LOSO**.